# 🧹 Limpiador de Tablas — Notebook interactivo

Analiza y corrige problemas de calidad de datos (valores faltantes, duplicados,
errores de tipo y valores atípicos) de forma **paso a paso**, con widgets para
elegir la fuente, el método de detección y la acción por cada tipo de hallazgo.

Ejecute las celdas en orden. No requiere tocar el código.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import matplotlib.pyplot as plt

from data_cleaner import (
    load_table, analizar, limpiar, DEFAULT_CONFIG,
    construir_reporte, exportar_reporte_excel, exportar,
)

OPCIONES_ACCION = {
    "faltante": ["reemplazar_mediana", "reemplazar_media", "reemplazar_moda",
                 "valor_fijo", "eliminar_fila", "marcar_solo"],
    "duplicado": ["eliminar_fila", "marcar_solo"],
    "atipico": ["limitar", "reemplazar_mediana", "reemplazar_media",
                "eliminar_fila", "marcar_solo"],
    "tipo_invalido": ["eliminar_fila", "valor_fijo", "marcar_solo"],
}
NOMBRES_TIPO = {
    "faltante": "Valores faltantes",
    "duplicado": "Filas duplicadas",
    "atipico": "Valores atípicos",
    "tipo_invalido": "Errores de tipo",
}
print("Librerías cargadas.")

## 1. Cargar los datos

Suba un archivo CSV/Excel, o use los datos de ejemplo incluidos.

In [ ]:
estado = {"df": None, "nombre_fuente": None, "resultado": None,
          "df_limpio": None, "registro": None, "tablas_reporte": None}

cargador = widgets.FileUpload(accept=".csv,.xlsx,.xls", multiple=False, description="Subir archivo")
btn_ejemplo = widgets.Button(description="Usar ejemplo_datos.csv", button_style="info")
salida_carga = widgets.Output()


def _cargar_desde_upload(change):
    if not cargador.value:
        return
    archivo = list(cargador.value.values())[0] if isinstance(cargador.value, dict) else cargador.value[0]
    nombre = archivo["metadata"]["name"] if isinstance(archivo, dict) else archivo.name
    contenido = archivo["content"] if isinstance(archivo, dict) else archivo.content
    with salida_carga:
        clear_output()
        try:
            if nombre.lower().endswith(".csv"):
                df = pd.read_csv(io.BytesIO(bytes(contenido)))
            else:
                df = pd.read_excel(io.BytesIO(bytes(contenido)))
        except Exception as exc:
            print(f"Error al leer el archivo: {exc}")
            return
        estado["df"] = df
        estado["nombre_fuente"] = nombre
        print(f"Cargado: {nombre} ({len(df)} filas x {len(df.columns)} columnas)")
        display(df.head())


def _cargar_ejemplo(_):
    with salida_carga:
        clear_output()
        df = load_table("ejemplo_datos.csv", kind="csv")
        estado["df"] = df
        estado["nombre_fuente"] = "ejemplo_datos.csv"
        print(f"Cargado: ejemplo_datos.csv ({len(df)} filas x {len(df.columns)} columnas)")
        display(df.head())


cargador.observe(_cargar_desde_upload, names="value")
btn_ejemplo.on_click(_cargar_ejemplo)

display(widgets.HBox([cargador, btn_ejemplo]), salida_carga)

## 2. Analizar la tabla

Elija el método de detección de valores atípicos y ejecute el análisis.

In [ ]:
metodo_atipicos = widgets.Dropdown(
    options=[("IQR (rango intercuartílico)", "iqr"),
             ("Z-score", "zscore"),
             ("Ambos (fusionados)", "ambos")],
    value="iqr", description="Método:",
)
btn_analizar = widgets.Button(description="🔍 Analizar", button_style="primary")
salida_analisis = widgets.Output()


def _analizar(_):
    with salida_analisis:
        clear_output()
        if estado["df"] is None:
            print("Primero cargue un archivo en el paso 1.")
            return
        resultado = analizar(estado["df"], metodo_atipicos=metodo_atipicos.value)
        estado["resultado"] = resultado
        estado["df_limpio"] = None

        print(f"Filas analizadas: {resultado.filas_analizadas}")
        print(f"Columnas analizadas: {resultado.columnas_analizadas}")
        print(f"Total de hallazgos: {len(resultado.issues)}")

        por_tipo = resultado.por_tipo()
        if not por_tipo:
            print("\n✅ No se encontraron problemas de calidad de datos.")
            return

        for tipo, cantidad in por_tipo.items():
            print(f"  {NOMBRES_TIPO.get(tipo, tipo)}: {cantidad}")

        fig, ax = plt.subplots(figsize=(5, 3))
        ax.bar(list(por_tipo.keys()), list(por_tipo.values()), color="#4C72B0")
        ax.set_title("Hallazgos por tipo")
        ax.set_ylabel("cantidad")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()

        detalle = pd.DataFrame([
            {"tipo": i.tipo, "columna": i.columna or "(fila completa)",
             "fila": i.fila, "valor_original": i.valor_original, "detalle": i.detalle}
            for i in resultado.issues
        ])
        display(detalle)


btn_analizar.on_click(_analizar)
display(widgets.HBox([metodo_atipicos, btn_analizar]), salida_analisis)

## 3. Configurar la corrección

Para cada tipo de problema encontrado, elija la acción a aplicar. Si elige **valor_fijo**, aparecerá un campo de texto por columna afectada (recuerde llenarlo: si queda vacío, esa columna no se corrige).

In [ ]:
panel_config = widgets.VBox()
salida_config = widgets.Output()


def _construir_panel_config():
    resultado = estado["resultado"]
    if resultado is None or not resultado.issues:
        panel_config.children = [widgets.Label("Ejecute el análisis (paso 2) primero.")]
        return

    filas = []
    accion_widgets = {}
    valor_fijo_widgets = {}

    for tipo, cantidad in resultado.por_tipo().items():
        if tipo not in OPCIONES_ACCION:
            continue
        opciones = OPCIONES_ACCION[tipo]
        defecto = DEFAULT_CONFIG.get(tipo, opciones[0])

        dd = widgets.Dropdown(options=opciones, value=defecto,
                               description=f"{NOMBRES_TIPO.get(tipo, tipo)} ({cantidad}):",
                               style={"description_width": "230px"}, layout=widgets.Layout(width="480px"))
        accion_widgets[tipo] = dd

        columnas_afectadas = sorted({i.columna for i in resultado.issues if i.tipo == tipo and i.columna})
        caja_valores = widgets.HBox()

        def _actualizar(change, tipo=tipo, cols=columnas_afectadas, caja=caja_valores):
            if change["new"] == "valor_fijo":
                nuevos = []
                for col in cols:
                    txt = widgets.Text(description=col, placeholder="valor de reemplazo",
                                        layout=widgets.Layout(width="220px"))
                    valor_fijo_widgets[f"{tipo}::{col}"] = txt
                    nuevos.append(txt)
                caja.children = nuevos
            else:
                caja.children = []
                for col in cols:
                    valor_fijo_widgets.pop(f"{tipo}::{col}", None)

        dd.observe(_actualizar, names="value")
        if defecto == "valor_fijo":
            _actualizar({"new": "valor_fijo"})

        filas.append(widgets.VBox([dd, caja_valores]))

    panel_config.children = filas
    estado["_accion_widgets"] = accion_widgets
    estado["_valor_fijo_widgets"] = valor_fijo_widgets


btn_construir = widgets.Button(description="⚙️ Cargar opciones de configuración")
btn_construir.on_click(lambda _: _construir_panel_config())
display(btn_construir, panel_config)

## 4. Limpiar y generar el reporte

In [ ]:
btn_limpiar = widgets.Button(description="🧽 Limpiar tabla", button_style="success")
salida_limpieza = widgets.Output()


def _limpiar(_):
    with salida_limpieza:
        clear_output()
        resultado = estado["resultado"]
        if resultado is None:
            print("Primero analice la tabla (paso 2).")
            return
        if not resultado.issues:
            print("No hay hallazgos que limpiar.")
            return

        accion_widgets = estado.get("_accion_widgets", {})
        valor_fijo_widgets = estado.get("_valor_fijo_widgets", {})
        config = {tipo: dd.value for tipo, dd in accion_widgets.items()}

        valores_fijos = {}
        faltan = []
        for tipo, accion in config.items():
            if accion != "valor_fijo":
                continue
            columnas_afectadas = sorted({i.columna for i in resultado.issues if i.tipo == tipo and i.columna})
            for col in columnas_afectadas:
                txt = valor_fijo_widgets.get(f"{tipo}::{col}")
                valor = txt.value.strip() if txt else ""
                if valor == "":
                    faltan.append(f"{NOMBRES_TIPO.get(tipo, tipo)} → columna '{col}'")
                else:
                    valores_fijos[col] = valor

        if faltan:
            print("⚠️ Complete el valor fijo de reemplazo para:")
            for f in faltan:
                print(f"  - {f}")
            return

        df_limpio, registro = limpiar(estado["df"], resultado.issues, config=config, valores_fijos=valores_fijos)
        tablas_reporte = construir_reporte(resultado, registro, nombre_fuente=estado["nombre_fuente"] or "")

        estado["df_limpio"] = df_limpio
        estado["registro"] = registro
        estado["tablas_reporte"] = tablas_reporte

        n_original = len(estado["df"])
        print(f"✅ Filas finales: {len(df_limpio)} (originales: {n_original})")
        display(df_limpio.head(20))


btn_limpiar.on_click(_limpiar)
display(btn_limpiar, salida_limpieza)

## 5. Descargar resultados

In [ ]:
btn_guardar = widgets.Button(description="💾 Guardar en disco (carpeta salida/)", button_style="warning")
salida_guardado = widgets.Output()


def _guardar(_):
    with salida_guardado:
        clear_output()
        if estado["df_limpio"] is None:
            print("Primero limpie la tabla (paso 4).")
            return
        import os
        os.makedirs("salida", exist_ok=True)
        ruta_limpio = "salida/datos_limpios.xlsx"
        ruta_reporte = "salida/reporte_calidad_datos.xlsx"
        exportar(estado["df_limpio"], ruta_limpio, kind="excel")
        exportar_reporte_excel(estado["tablas_reporte"], ruta_reporte)
        print(f"Guardado:\n  {ruta_limpio}\n  {ruta_reporte}")


btn_guardar.on_click(_guardar)
display(btn_guardar, salida_guardado)